# Random Forest LOSO Baseline: Binary PREICTAL vs INTERICTAL

**Task scope.** Second modeling experiment on the patient-specific LOSO
pipeline, mirroring `EEG_logistic_loso_binary.ipynb` as closely as
reasonably possible. **Only the estimator changes** -- Random Forest
replaces Logistic Regression. No preprocessing/labeling/normalization/LOSO
redesign, no hyperparameter tuning, no new patients/horizons/features.

**Reused, unmodified inputs (verbatim, credited, copied from the already-
validated notebooks -- not reimplemented):**
- `outputs/labels_binary_preictal.parquet` -- RAW (unscaled) qEEG features +
  binary labels for all three horizons (`EEG_binary_preictal_labeling.ipynb`).
- Fold-construction functions (`compute_seizure_eligibility`,
  `assign_fold_context`, `fit_patient_scalers`,
  `transform_with_patient_scalers`, `build_loso_fold`, `validate_loso_fold`)
  copied verbatim from `EEG_logistic_loso_binary.ipynb` (itself copied from
  `EEG_loso_evaluation_splitting.ipynb`). Folds are rebuilt fresh here and
  cross-checked against `outputs/loso_fold_manifest.parquet` for exact
  agreement, exactly as the Logistic Regression notebook does.
- Metric functions (`fold_seizure_event_metrics`, `fold_epoch_metrics`,
  `pool_epoch_metrics`, `pool_event_metrics`) copied verbatim from
  `EEG_logistic_loso_binary.ipynb` / `EEG_seizure_prediction_metrics.ipynb`.

**New output:** everything under `outputs/modeling/random_forest_loso_binary/`.
Nothing existing is modified, including `outputs/modeling/logistic_loso_binary/`,
which is only ever *read* (Section 15, for the LR vs RF comparison).


## 1. Input pipeline verification

Same chain as the Logistic Regression notebook, with the estimator swapped:

```text
RAW/unscaled qEEG features (outputs/labels_binary_preictal.parquet)
        |
binary_label_h{5,15,30} (already computed, reused as-is)
        |
patient-specific LOSO fold (one held-out seizure at a time)
        |
fit StandardScaler on this fold's TRAIN rows only
        |
transform TRAIN and TEST with that TRAIN-only scaler
        |
RandomForestClassifier, fit on TRAIN only
        |
predict_proba on TEST -> P(PREICTAL)
        |
seizure-prediction metric framework (same functions as the LR notebook)
```

Deliberately **not** using `outputs/features_patient_scaled.parquet`, for the
same leakage reason documented in `EEG_logistic_loso_binary.ipynb` /
`EEG_loso_evaluation_splitting.ipynb`: that file's scaled columns come from
one fixed recording-level split, not this fold's own TRAIN-only scaler.


In [1]:

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, precision_recall_curve

print("=" * 70)
print("RANDOM FOREST LOSO BINARY BASELINE -- SETUP")
print("=" * 70)
print(f"numpy : {np.__version__}")
print(f"pandas: {pd.__version__}")
import sklearn
print(f"sklearn: {sklearn.__version__}")

repo_root = Path.cwd()
output_dir = repo_root / "outputs"
run_dir = output_dir / "modeling" / "random_forest_loso_binary"
plot_dir = run_dir / "plots"
run_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)

lr_run_dir = output_dir / "modeling" / "logistic_loso_binary"
if not lr_run_dir.exists():
    raise FileNotFoundError(f"{lr_run_dir} not found -- run EEG_logistic_loso_binary.ipynb first (reference baseline).")

in_path = output_dir / "labels_binary_preictal.parquet"
if not in_path.exists():
    raise FileNotFoundError(f"{in_path} not found -- run EEG_binary_preictal_labeling.ipynb first.")
df = pd.read_parquet(in_path)
print(f"\nLoaded {in_path.name}: shape={df.shape}")
print(f"Source table confirmed: RAW/unscaled qEEG features (no '_scaled' columns present: "
      f"{not any(c.endswith('_scaled') for c in df.columns)})")

PATIENT_COL = "patient_id"
HORIZONS = [5, 15, 30]
NON_FEATURE_COLS = {
    "patient_id", "subject_id", "recording_id", "dataset", "epoch_idx",
    "t_start_sec", "t_center_sec", "t_end_sec",
    "y_seizure", "seizure_id", "next_seizure_id", "next_seizure_start_sec",
    "time_to_next_seizure_sec", "previous_seizure_id",
    "previous_seizure_end_sec", "time_since_previous_seizure_sec",
    "binary_label_h5", "binary_label_h15", "binary_label_h30",
    "exclusion_reason_h5", "exclusion_reason_h15", "exclusion_reason_h30",
}
FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE_COLS]
PATIENTS = sorted(df[PATIENT_COL].unique())
print(f"\nFeature columns used ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Patients (modeled SEPARATELY, never pooled): {PATIENTS}")

EPOCH_SEC = float((df["t_end_sec"] - df["t_start_sec"]).round(3).mode().iloc[0])
assert abs(EPOCH_SEC - 30.0) < 1e-6, f"Expected 30s epochs (project_state.md 5.1), got {EPOCH_SEC}"
print(f"Epoch duration (from data): {EPOCH_SEC}s")

RANDOM_STATE = 42
THRESHOLD = 0.5
print(f"\nRANDOM_STATE={RANDOM_STATE}, decision THRESHOLD={THRESHOLD} (fixed, not tuned on TEST)")

# Sanity check: the feature set used here must be IDENTICAL to the LR baseline's.
import json as _json_check
with open(lr_run_dir / "experiment_config.json") as _f:
    _lr_config = _json_check.load(_f)
assert FEATURE_COLS == _lr_config["feature_cols"], "FEATURE_COLS differ from the LR baseline -- fair comparison requires identical features"
assert HORIZONS == _lr_config["horizons_minutes"], "HORIZONS differ from the LR baseline"
assert PATIENTS == _lr_config["patients"], "PATIENTS differ from the LR baseline"
print("\n[OK] FEATURE_COLS, HORIZONS, and PATIENTS are byte-for-byte identical to the LR baseline's experiment_config.json.")


RANDOM FOREST LOSO BINARY BASELINE -- SETUP
numpy : 2.4.6
pandas: 3.0.5
sklearn: 1.9.1



Loaded labels_binary_preictal.parquet: shape=(2040, 36)
Source table confirmed: RAW/unscaled qEEG features (no '_scaled' columns present: True)

Feature columns used (14): ['std', 'var', 'rms', 'delta', 'theta', 'alpha', 'beta', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'theta_alpha_ratio', 'delta_beta_ratio', 'entropy']
Patients (modeled SEPARATELY, never pooled): ['chb12', 'chb24']
Epoch duration (from data): 30.0s

RANDOM_STATE=42, decision THRESHOLD=0.5 (fixed, not tuned on TEST)

[OK] FEATURE_COLS, HORIZONS, and PATIENTS are byte-for-byte identical to the LR baseline's experiment_config.json.


## 2. Fold-construction functions

Copied verbatim from `EEG_logistic_loso_binary.ipynb` (which copied them
verbatim from `EEG_loso_evaluation_splitting.ipynb`, itself building on
`EEG_patient_normalization.ipynb`). Not reimplemented -- this notebook's
folds are byte-for-byte the same construction already leakage-validated
(57/57 folds, 9 checks each) -- verified again below by cross-checking
against the saved manifest, exactly as the LR notebook does.


In [2]:

def compute_seizure_eligibility(df, patient_id, horizon_minutes, seizure_ids):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_loso_evaluation_splitting.ipynb.)
    Returns (eligible_seizure_ids, ineligible_rows) for one patient/horizon.'''
    lab_col = f"binary_label_h{horizon_minutes}"
    reason_col = f"exclusion_reason_h{horizon_minutes}"
    horizon_sec = horizon_minutes * 60.0
    pdf = df[df["patient_id"] == patient_id]

    eligible, ineligible = [], []
    for sid in seizure_ids:
        n_preictal = int(((pdf[lab_col] == "PREICTAL") & (pdf["next_seizure_id"] == sid)).sum())
        if n_preictal > 0:
            eligible.append(sid)
            continue
        candidates = pdf[
            (pdf["next_seizure_id"] == sid)
            & (pdf["time_to_next_seizure_sec"] >= 0)
            & (pdf["time_to_next_seizure_sec"] <= horizon_sec)
        ]
        if len(candidates) == 0:
            reason = "no candidate epochs within horizon (insufficient recording history before onset)"
        else:
            reason_counts = candidates[reason_col].value_counts()
            top_reason = reason_counts.idxmax() if len(reason_counts) else "unknown"
            reason = f"all {len(candidates)} candidate epoch(s) excluded (dominant reason: {top_reason})"
        ineligible.append({"seizure_id": sid, "reason": reason})
    return eligible, ineligible


def assign_fold_context(df, horizon_minutes):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_loso_evaluation_splitting.ipynb.)'''
    lab_col = f"binary_label_h{horizon_minutes}"
    fold_context = pd.Series(None, index=df.index, dtype=object)

    is_preictal = (df[lab_col] == "PREICTAL").to_numpy()
    is_interictal = (df[lab_col] == "INTERICTAL").to_numpy()

    assert df.loc[is_preictal, "next_seizure_id"].notna().all(), \
        "A PREICTAL epoch has no next_seizure_id -- cannot assign fold context"
    fold_context.loc[is_preictal] = df.loc[is_preictal, "next_seizure_id"]

    inter_next = df.loc[is_interictal, "next_seizure_id"]
    inter_prev = df.loc[is_interictal, "previous_seizure_id"]
    inter_context = inter_next.where(inter_next.notna(), inter_prev)
    assert inter_context.notna().all(), \
        "An INTERICTAL epoch has neither a next nor a previous seizure in its recording"
    fold_context.loc[is_interictal] = inter_context

    return fold_context


def fit_patient_scalers(df, patient_col, feature_cols, train_mask):
    '''(Verbatim from EEG_patient_normalization.ipynb, via EEG_logistic_loso_binary.ipynb.)'''
    scalers = {}
    train_mask = np.asarray(train_mask)
    for pid in df.loc[train_mask, patient_col].unique():
        sel = train_mask & (df[patient_col] == pid).to_numpy()
        n = int(sel.sum())
        if n == 0:
            raise ValueError(f"Patient {pid!r} has zero TRAIN rows -- cannot fit a scaler.")
        assert df.loc[sel, patient_col].nunique() == 1
        scaler = StandardScaler()
        scaler.fit(df.loc[sel, feature_cols])
        scalers[pid] = scaler
    return scalers


def transform_with_patient_scalers(df, patient_col, feature_cols, scalers):
    '''(Verbatim from EEG_patient_normalization.ipynb, via EEG_logistic_loso_binary.ipynb.)'''
    out = df.copy()
    scaled = np.full((len(df), len(feature_cols)), np.nan)
    covered = np.zeros(len(df), dtype=bool)
    for pid, scaler in scalers.items():
        sel = (df[patient_col] == pid).to_numpy()
        if sel.any():
            scaled[sel, :] = scaler.transform(df.loc[sel, feature_cols])
            covered |= sel
    if not covered.all():
        missing_pids = sorted(df.loc[~covered, patient_col].unique())
        raise ValueError(f"No fitted scaler available for patients: {missing_pids}")
    scaled_cols = [f"{c}_scaled" for c in feature_cols]
    out[scaled_cols] = scaled
    return out, scaled_cols


def build_loso_fold(patient_df, horizon_minutes, held_out_seizure_id, fold_context, feature_cols, patient_col="patient_id"):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_loso_evaluation_splitting.ipynb.)
    One LOSO fold: ONE patient, ONE horizon, ONE held-out seizure.'''
    lab_col = f"binary_label_h{horizon_minutes}"
    labeled_mask = patient_df[lab_col].notna().to_numpy()
    ctx = fold_context.loc[patient_df.index].to_numpy()

    test_mask = labeled_mask & (ctx == held_out_seizure_id)
    train_mask = labeled_mask & (ctx != held_out_seizure_id)

    scalers = fit_patient_scalers(patient_df, patient_col, feature_cols, train_mask)
    scaled_df, scaled_cols = transform_with_patient_scalers(patient_df, patient_col, feature_cols, scalers)

    pid = patient_df[patient_col].iloc[0]
    return {
        "patient_id": pid,
        "horizon_minutes": horizon_minutes,
        "held_out_seizure_id": held_out_seizure_id,
        "lab_col": lab_col,
        "train_idx": patient_df.index[train_mask],
        "test_idx": patient_df.index[test_mask],
        "scaler": scalers[pid],
        "scaled_df": scaled_df,
        "scaled_cols": scaled_cols,
    }


def validate_loso_fold(fold, patient_df, feature_cols):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_loso_evaluation_splitting.ipynb.) 9 leakage checks.'''
    train_idx, test_idx = fold["train_idx"], fold["test_idx"]
    lab_col = fold["lab_col"]
    held_out = fold["held_out_seizure_id"]

    assert len(set(train_idx) & set(test_idx)) == 0, "TRAIN/TEST indices overlap"

    train_next = patient_df.loc[train_idx, "next_seizure_id"]
    train_preictal = patient_df.loc[train_idx, lab_col] == "PREICTAL"
    assert not (train_preictal & (train_next == held_out)).any(), \
        f"Held-out seizure {held_out} has a PREICTAL epoch leaking into TRAIN"

    scaler = fold["scaler"]
    assert int(scaler.n_samples_seen_) == len(train_idx), \
        f"scaler.n_samples_seen_={scaler.n_samples_seen_} != len(train_idx)={len(train_idx)}"
    manual_mean = patient_df.loc[train_idx, feature_cols].to_numpy().mean(axis=0)
    assert np.allclose(manual_mean, scaler.mean_), "scaler mean does not match TRAIN-only recomputation"

    combined = pd.concat([patient_df.loc[train_idx, "patient_id"], patient_df.loc[test_idx, "patient_id"]])
    assert combined.nunique() == 1, "More than one patient present in a single LOSO fold"

    assert list(scaler.feature_names_in_) == list(feature_cols), "Feature column order mismatch at fit time"

    assert patient_df.loc[train_idx, lab_col].isin(["PREICTAL", "INTERICTAL"]).all()
    assert patient_df.loc[test_idx, lab_col].isin(["PREICTAL", "INTERICTAL"]).all()

    assert (patient_df.loc[train_idx, "y_seizure"] == 0).all(), "A seizure-overlapping epoch is in TRAIN"
    assert (patient_df.loc[test_idx, "y_seizure"] == 0).all(), "A seizure-overlapping epoch is in TEST"

    all_idx = train_idx.union(test_idx)
    for col in ["recording_id", "epoch_idx", "t_start_sec", "t_end_sec"]:
        assert patient_df.loc[all_idx, col].notna().all(), f"Metadata column {col} has nulls in this fold"

    return True


print("[OK] Fold-construction functions reused verbatim from EEG_logistic_loso_binary.ipynb.")


[OK] Fold-construction functions reused verbatim from EEG_logistic_loso_binary.ipynb.


## 3. Metric functions

Copied verbatim from `EEG_logistic_loso_binary.ipynb` /
`EEG_seizure_prediction_metrics.ipynb`. Identical metric definitions,
identical NaN behavior for zero-TEST-interictal folds and single-class TEST
folds -- Random Forest predictions must pass through the exact same
evaluation code as Logistic Regression for the comparison to be fair.


In [3]:

REQUIRED_PRED_COLS = [
    "patient_id", "horizon_minutes", "fold_id", "held_out_seizure_id",
    "recording_id", "epoch_idx", "t_start_sec", "t_end_sec",
    "time_to_next_seizure_sec", "y_true", "y_prob", "y_pred",
]

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, average_precision_score,
)


def validate_predictions_df(df, require_single_fold=False):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)'''
    missing = [c for c in REQUIRED_PRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"predictions_df missing required columns: {missing}")
    if df["y_true"].isna().any() or not set(df["y_true"].unique()) <= {0, 1}:
        raise ValueError("y_true must be exactly {0, 1} with no NaNs")
    if df["y_pred"].isna().any() or not set(df["y_pred"].unique()) <= {0, 1}:
        raise ValueError("y_pred must be exactly {0, 1} with no NaNs")
    if df["y_prob"].isna().any() or (df["y_prob"] < 0).any() or (df["y_prob"] > 1).any():
        raise ValueError("y_prob must be in [0, 1] with no NaNs")
    if require_single_fold:
        key_cols = ["patient_id", "horizon_minutes", "held_out_seizure_id"]
        if df[key_cols].drop_duplicates().shape[0] != 1:
            raise ValueError("fold_df must contain exactly one (patient, horizon, held-out seizure)")
    return True


def _count_alarm_events(sorted_epoch_idx, sorted_y_pred):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)'''
    n = len(sorted_epoch_idx)
    if n == 0:
        return 0
    epoch_idx = np.asarray(sorted_epoch_idx)
    y_pred = np.asarray(sorted_y_pred)
    new_block = np.concatenate(([True], np.diff(epoch_idx) != 1))
    block_id = np.cumsum(new_block)
    n_events = 0
    for b in np.unique(block_id):
        p = y_pred[block_id == b]
        new_run = np.concatenate(([True], np.diff(p) != 0))
        run_id = np.cumsum(new_run)
        for r in np.unique(run_id):
            if p[run_id == r][0] == 1:
                n_events += 1
    return n_events


def fold_seizure_event_metrics(fold_df, epoch_sec=EPOCH_SEC):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)
    Primary metrics for ONE LOSO fold.'''
    validate_predictions_df(fold_df, require_single_fold=True)

    pre = fold_df[fold_df["y_true"] == 1].sort_values("t_start_sec")
    inter = fold_df[fold_df["y_true"] == 0].sort_values(["recording_id", "epoch_idx"])

    n_preictal_alarms = int((pre["y_pred"] == 1).sum())
    detected = int(n_preictal_alarms > 0)

    if detected:
        alarm_epochs = pre[pre["y_pred"] == 1]
        first_alarm = alarm_epochs.loc[alarm_epochs["time_to_next_seizure_sec"].idxmax()]
        time_in_warning_sec = float(first_alarm["time_to_next_seizure_sec"])
    else:
        time_in_warning_sec = np.nan

    n_false_alarm_events = _count_alarm_events(inter["epoch_idx"].to_numpy(), inter["y_pred"].to_numpy())
    n_interictal_epochs = len(inter)
    interictal_hours = n_interictal_epochs * epoch_sec / 3600.0
    false_alarms_per_hour = (
        n_false_alarm_events / interictal_hours if interictal_hours > 0 else np.nan
    )

    return {
        "n_test_preictal": len(pre),
        "n_test_interictal": n_interictal_epochs,
        "seizure_detected": detected,
        "n_preictal_alarm_epochs": n_preictal_alarms,
        "time_in_warning_sec": time_in_warning_sec,
        "n_false_alarm_events": n_false_alarm_events,
        "interictal_hours": interictal_hours,
        "false_alarms_per_hour": false_alarms_per_hour,
    }


def fold_epoch_metrics(fold_df):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)
    Secondary metrics; works on any epoch set.'''
    validate_predictions_df(fold_df, require_single_fold=False)

    y_true = fold_df["y_true"].to_numpy()
    y_pred = fold_df["y_pred"].to_numpy()
    y_prob = fold_df["y_prob"].to_numpy()

    n_classes_true = len(np.unique(y_true))

    accuracy = float(accuracy_score(y_true, y_pred))
    precision = float(precision_score(y_true, y_pred, pos_label=1, zero_division=np.nan))
    recall = float(recall_score(y_true, y_pred, pos_label=1, zero_division=np.nan))
    f1 = float(f1_score(y_true, y_pred, pos_label=1, zero_division=np.nan))

    if n_classes_true < 2:
        auc_pr = np.nan
        auc_pr_undefined_reason = "only one class present in y_true (AUC-PR is not meaningful)"
    else:
        auc_pr = float(average_precision_score(y_true, y_prob))
        auc_pr_undefined_reason = None

    return {
        "n_epochs": len(fold_df),
        "n_classes_in_test": n_classes_true,
        "accuracy": accuracy,
        "precision_preictal": precision,
        "recall_preictal": recall,
        "f1_preictal": f1,
        "auc_pr": auc_pr,
        "auc_pr_undefined_reason": auc_pr_undefined_reason,
    }


def pool_epoch_metrics(pooled_epochs_df):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)'''
    return fold_epoch_metrics(pooled_epochs_df)


def pool_event_metrics(fold_event_metrics_df):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)'''
    n_folds = len(fold_event_metrics_df)
    n_detected = int(fold_event_metrics_df["seizure_detected"].sum())
    seizure_sensitivity = n_detected / n_folds if n_folds > 0 else np.nan

    total_fa_events = int(fold_event_metrics_df["n_false_alarm_events"].sum())
    total_interictal_hours = float(fold_event_metrics_df["interictal_hours"].sum())
    pooled_fa_per_hour = (
        total_fa_events / total_interictal_hours if total_interictal_hours > 0 else np.nan
    )

    detected_rows = fold_event_metrics_df[fold_event_metrics_df["seizure_detected"] == 1]
    tiw = detected_rows["time_in_warning_sec"].dropna()
    if len(tiw) > 0:
        tiw_mean, tiw_median, tiw_min, tiw_max = (
            float(tiw.mean()), float(tiw.median()), float(tiw.min()), float(tiw.max())
        )
    else:
        tiw_mean = tiw_median = tiw_min = tiw_max = np.nan

    return {
        "n_folds": n_folds,
        "n_seizures_detected": n_detected,
        "seizure_sensitivity": seizure_sensitivity,
        "total_false_alarm_events": total_fa_events,
        "total_interictal_hours": total_interictal_hours,
        "false_alarms_per_hour": pooled_fa_per_hour,
        "n_detected_with_time_in_warning": int(len(tiw)),
        "time_in_warning_sec_mean": tiw_mean,
        "time_in_warning_sec_median": tiw_median,
        "time_in_warning_sec_min": tiw_min,
        "time_in_warning_sec_max": tiw_max,
    }


print("[OK] Metric functions reused verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.")


[OK] Metric functions reused verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.


## 4. TRAIN class-imbalance inspection -> class_weight decision (Random Forest equivalent)

The Logistic Regression baseline used `class_weight="balanced"`, decided
after inspecting per-fold TRAIN prevalence in `outputs/loso_fold_manifest.parquet`
(median PREICTAL prevalence 0.514 at H=5, ranging 0.251-0.826 across
folds/horizons; 36/57 folds outside [0.4, 0.6]). That imbalance is a
property of the data/fold construction, not of the estimator, so it applies
identically here.

**RF equivalent choice:** `RandomForestClassifier` supports `class_weight`
directly, with two options: `"balanced"` (weights computed once from the
whole TRAIN `y`, exactly mirroring what `LogisticRegression` does) and
`"balanced_subsample"` (recomputed independently for each tree's bootstrap
resample). `"balanced"` is chosen here as the closer analogue to the LR
baseline's `class_weight="balanced"` -- both derive one fixed weighting from
the fold's full TRAIN label vector, so the imbalance-handling mechanism is
as comparable as the two algorithms allow. Like LR's `class_weight`, this is
computed by sklearn from whatever `y` is passed to `.fit()`, i.e. strictly
from that fold's own TRAIN labels -- never TEST.

No other hyperparameter is tuned per fold or from TEST data. To keep this a
genuinely simple, reproducible first baseline (mirroring LR leaving `C` at
its sklearn default), every RF hyperparameter besides `class_weight`,
`random_state`, and `n_jobs` (parallelism only, does not affect results) is
left at its sklearn default (`n_estimators=100`, `max_depth=None`,
`min_samples_split=2`, `min_samples_leaf=1`, `max_features="sqrt"`, ...).


In [4]:

manifest_path = output_dir / "loso_fold_manifest.parquet"
if not manifest_path.exists():
    raise FileNotFoundError(f"{manifest_path} not found -- run EEG_loso_evaluation_splitting.ipynb first.")
existing_manifest = pd.read_parquet(manifest_path)

existing_manifest["train_prevalence_preictal"] = (
    existing_manifest["train_preictal_count"]
    / (existing_manifest["train_preictal_count"] + existing_manifest["train_interictal_count"])
)
print("TRAIN class balance across all 57 existing LOSO folds (from the validated manifest, same as LR notebook):")
print(existing_manifest.groupby("horizon_minutes")["train_prevalence_preictal"].describe().to_string())

n_imbalanced = (
    (existing_manifest["train_prevalence_preictal"] < 0.4)
    | (existing_manifest["train_prevalence_preictal"] > 0.6)
).sum()
print(f"\nFolds with TRAIN PREICTAL prevalence outside [0.4, 0.6]: {n_imbalanced}/{len(existing_manifest)}")

print(
    "\n[DECISION] class_weight='balanced' will be used for every fold's RandomForestClassifier -- the "
    "closest RF analogue to the LR baseline's class_weight='balanced' (see markdown above). Computed by "
    "sklearn strictly from each fold's own TRAIN y at .fit() time -- never TEST. All other hyperparameters "
    "left at sklearn defaults; nothing is tuned per fold."
)

RF_PARAMS = dict(
    n_estimators=100,       # sklearn default -- deliberately not tuned/enlarged for this first baseline
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,               # parallelism only, does not affect the fitted model or its predictions
)
print(f"\nFixed RandomForestClassifier config for every fold: {RF_PARAMS}")


TRAIN class balance across all 57 existing LOSO folds (from the validated manifest, same as LR notebook):
                 count      mean       std       min       25%       50%       75%       max
horizon_minutes                                                                             
5                 19.0  0.331387  0.110302  0.250559  0.254189  0.281407  0.404962  0.519481
15                19.0  0.552852  0.106113  0.467409  0.480496  0.506897  0.624188  0.753333
30                19.0  0.667340  0.081939  0.594660  0.614906  0.630491  0.720069  0.826291

Folds with TRAIN PREICTAL prevalence outside [0.4, 0.6]: 36/57

[DECISION] class_weight='balanced' will be used for every fold's RandomForestClassifier -- the closest RF analogue to the LR baseline's class_weight='balanced' (see markdown above). Computed by sklearn strictly from each fold's own TRAIN y at .fit() time -- never TEST. All other hyperparameters left at sklearn defaults; nothing is tuned per fold.

Fixed RandomFo

## 5. Patient-specific modeling invariant

Identical to the LR notebook. CHB12 and CHB24 are **never** pooled into one
training model. Every fold is built from `patient_df = df[df.patient_id ==
pid]` before any TRAIN/TEST split or scaler fit, structurally enforced by
`build_loso_fold`/`validate_loso_fold` (leakage check 5: exactly one patient
per fold). The loop below asserts it again explicitly, per-fold, before
fitting any model.


In [5]:

print("[OK] Patient-specific invariant will be asserted explicitly for every fold in the main loop below "
      "(Section 6): each fold's model is fit ONLY on that one patient's TRAIN rows, and separately for "
      "chb12 and chb24 -- never pooled.")


[OK] Patient-specific invariant will be asserted explicitly for every fold in the main loop below (Section 6): each fold's model is fit ONLY on that one patient's TRAIN rows, and separately for chb12 and chb24 -- never pooled.


## 6. Main experiment loop

Identical structure to the LR notebook's main loop (same fold construction,
same validation, same TRAIN/TEST split, same scaler). The **only** change is
the estimator: `RandomForestClassifier(**RF_PARAMS)` replaces
`LogisticRegression(**LOGREG_PARAMS)`. The same `DummyClassifier(strategy=
"most_frequent")` naive baseline is refit here (not reused from the LR run)
so this notebook is self-contained and reproducible on its own.


In [6]:

seizure_inventory = {}
for pid in PATIENTS:
    pdf = df[df[PATIENT_COL] == pid]
    seizure_inventory[pid] = sorted(pdf["seizure_id"].dropna().unique())

prediction_rows = []
importance_rows = []
fold_summary_rows = []
n_folds_built = 0

for H in HORIZONS:
    fold_context = assign_fold_context(df, H)
    lab_col = f"binary_label_h{H}"

    for pid in PATIENTS:
        patient_df = df[df[PATIENT_COL] == pid].copy()
        assert patient_df["patient_id"].nunique() == 1 and patient_df["patient_id"].iloc[0] == pid, \
            "Patient-specific invariant violated before fold construction"

        eligible, _ineligible = compute_seizure_eligibility(df, pid, H, seizure_inventory[pid])

        for fold_id, sid in enumerate(eligible, start=1):
            fold = build_loso_fold(patient_df, H, sid, fold_context, FEATURE_COLS)
            validate_loso_fold(fold, patient_df, FEATURE_COLS)
            n_folds_built += 1

            train_idx, test_idx = fold["train_idx"], fold["test_idx"]
            scaled_df, scaled_cols = fold["scaled_df"], fold["scaled_cols"]

            # Explicit patient-specific re-assertion (Section 5), not only inherited from validate_loso_fold.
            assert scaled_df.loc[train_idx.union(test_idx), "patient_id"].nunique() == 1
            assert scaled_df.loc[train_idx.union(test_idx), "patient_id"].iloc[0] == pid

            X_train = scaled_df.loc[train_idx, scaled_cols].to_numpy()
            y_train = (patient_df.loc[train_idx, lab_col] == "PREICTAL").astype(int).to_numpy()
            X_test = scaled_df.loc[test_idx, scaled_cols].to_numpy()
            y_test = (patient_df.loc[test_idx, lab_col] == "PREICTAL").astype(int).to_numpy()

            # --- Explicit leakage / train-only-fitting checks (beyond validate_loso_fold's 9) ---
            assert len(X_train) == len(y_train) == len(train_idx)
            assert len(X_test) == len(y_test) == len(test_idx)
            assert set(train_idx).isdisjoint(set(test_idx))
            n_train_pos, n_train_neg = int(y_train.sum()), int((1 - y_train).sum())
            if n_train_pos == 0 or n_train_neg == 0:
                raise RuntimeError(
                    f"H={H} {pid} fold {fold_id} ({sid}): TRAIN has only one class "
                    f"(pos={n_train_pos}, neg={n_train_neg}) -- cannot fit Random Forest. Stopping."
                )

            clf = RandomForestClassifier(**RF_PARAMS)
            clf.fit(X_train, y_train)  # TEST (X_test/y_test) not referenced until after this line
            dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
            dummy.fit(X_train, y_train)

            # Probability must correspond to the PREICTAL (1) class -- looked up via clf.classes_,
            # never assumed to be column index 1.
            pos_idx_clf = list(clf.classes_).index(1)
            y_prob_rf = clf.predict_proba(X_test)[:, pos_idx_clf]
            y_pred_rf = (y_prob_rf >= THRESHOLD).astype(int)

            pos_idx_dummy = list(dummy.classes_).index(1) if 1 in dummy.classes_ else None
            if pos_idx_dummy is not None:
                y_prob_dummy = dummy.predict_proba(X_test)[:, pos_idx_dummy]
            else:
                y_prob_dummy = np.zeros(len(X_test))  # dummy never predicts PREICTAL in TRAIN at all
            y_pred_dummy = (y_prob_dummy >= THRESHOLD).astype(int)

            meta_cols = ["recording_id", "epoch_idx", "t_start_sec", "t_end_sec", "time_to_next_seizure_sec"]
            test_meta = patient_df.loc[test_idx, meta_cols].reset_index(drop=True)

            for model_name, y_prob, y_pred in [
                ("random_forest", y_prob_rf, y_pred_rf),
                ("dummy_most_frequent", y_prob_dummy, y_pred_dummy),
            ]:
                rows = test_meta.copy()
                rows["patient_id"] = pid
                rows["horizon_minutes"] = H
                rows["fold_id"] = fold_id
                rows["held_out_seizure_id"] = sid
                rows["model"] = model_name
                rows["y_true"] = y_test
                rows["y_prob"] = y_prob
                rows["y_pred"] = y_pred
                prediction_rows.append(rows)

            importances = clf.feature_importances_
            importance_rows.append(pd.DataFrame({
                "patient_id": pid, "horizon_minutes": H, "fold_id": fold_id,
                "held_out_seizure_id": sid, "feature": FEATURE_COLS, "importance": importances,
            }))

            fold_summary_rows.append({
                "patient_id": pid, "horizon_minutes": H, "fold_id": fold_id,
                "held_out_seizure_id": sid,
                "n_train": len(train_idx), "n_test": len(test_idx),
                "train_preictal_count": n_train_pos, "train_interictal_count": n_train_neg,
                "train_prevalence_preictal": n_train_pos / len(train_idx),
            })

predictions_df = pd.concat(prediction_rows, ignore_index=True)
importances_df = pd.concat(importance_rows, ignore_index=True)
fold_summary_df = pd.DataFrame(fold_summary_rows)

print(f"[OK] Built, fit, and validated {n_folds_built} LOSO folds across {len(HORIZONS)} horizons "
      f"x {len(PATIENTS)} patients.")
print(f"predictions_df: {predictions_df.shape}")
print(f"importances_df: {importances_df.shape}")
print(f"fold_summary_df: {fold_summary_df.shape}")
assert n_folds_built == 57, f"Expected 57 LOSO folds (19 eligible seizures x 3 horizons), got {n_folds_built}"
print("\n[OK] No TEST label or TEST feature was read before the corresponding fold's clf.fit() call "
      "(clf.fit(X_train, y_train) executes before any X_test/y_test/predict_proba statement in the loop "
      "body above). No hyperparameter (n_estimators, class_weight formula, threshold) was chosen using "
      "TEST data -- all are fixed constants defined in Section 4 before this loop ran.")


[OK] Built, fit, and validated 57 LOSO folds across 3 horizons x 2 patients.
predictions_df: (4698, 13)
importances_df: (798, 6)
fold_summary_df: (57, 9)

[OK] No TEST label or TEST feature was read before the corresponding fold's clf.fit() call (clf.fit(X_train, y_train) executes before any X_test/y_test/predict_proba statement in the loop body above). No hyperparameter (n_estimators, class_weight formula, threshold) was chosen using TEST data -- all are fixed constants defined in Section 4 before this loop ran.


## 7. Cross-check against the existing validated fold manifest

Confirms this notebook reproduced *exactly* the same fold set already
validated in `EEG_loso_evaluation_splitting.ipynb` and used unchanged by
`EEG_logistic_loso_binary.ipynb` -- not a silently diverged reimplementation.


In [7]:

validate_predictions_df(predictions_df, require_single_fold=False)
print("[OK] predictions_df passes validate_predictions_df() (columns, value ranges).")

merged_check = fold_summary_df.merge(
    existing_manifest,
    on=["patient_id", "horizon_minutes", "held_out_seizure_id"],
    suffixes=("_thisrun", "_existing"),
)
assert len(merged_check) == len(fold_summary_df) == len(existing_manifest), (
    f"Fold count mismatch: this run={len(fold_summary_df)}, existing manifest={len(existing_manifest)}, "
    f"matched={len(merged_check)}"
)
assert (merged_check["n_train"] == merged_check["train_total"]).all()
assert (merged_check["n_test"] == merged_check["test_total"]).all()
assert (merged_check["train_preictal_count_thisrun"] == merged_check["train_preictal_count_existing"]).all()
assert (merged_check["train_interictal_count_thisrun"] == merged_check["train_interictal_count_existing"]).all()
print(f"[OK] All {len(merged_check)} folds match EEG_loso_evaluation_splitting.ipynb's saved manifest "
      f"exactly (train/test counts, preictal/interictal counts) -- fold construction reproduced identically "
      f"to the LR baseline.")

# Fold-for-fold cross-check against the LR baseline's own fold_summary (train/test composition must be
# IDENTICAL between the two experiments -- only the estimator may differ).
lr_fold_metrics = pd.read_csv(lr_run_dir / "fold_metrics.csv")
lr_fold_metrics_lr_only = lr_fold_metrics[lr_fold_metrics["model"] == "logistic_regression"].copy()
lr_fold_metrics_lr_only["n_test"] = lr_fold_metrics_lr_only["n_test_preictal"] + lr_fold_metrics_lr_only["n_test_interictal"]
lr_fold_keys = lr_fold_metrics_lr_only[
    ["patient_id", "horizon_minutes", "held_out_seizure_id", "n_train", "n_test",
     "train_preictal_count", "train_interictal_count"]
].reset_index(drop=True)
rf_fold_keys = fold_summary_df[
    ["patient_id", "horizon_minutes", "held_out_seizure_id", "n_train", "n_test",
     "train_preictal_count", "train_interictal_count"]
].reset_index(drop=True)
fold_cross = lr_fold_keys.merge(
    rf_fold_keys, on=["patient_id", "horizon_minutes", "held_out_seizure_id"], suffixes=("_lr", "_rf")
)
assert len(fold_cross) == len(lr_fold_keys) == len(rf_fold_keys) == 57
assert (fold_cross["n_train_lr"] == fold_cross["n_train_rf"]).all()
assert (fold_cross["n_test_lr"] == fold_cross["n_test_rf"]).all()
assert (fold_cross["train_preictal_count_lr"] == fold_cross["train_preictal_count_rf"]).all()
assert (fold_cross["train_interictal_count_lr"] == fold_cross["train_interictal_count_rf"]).all()
print(f"[OK] All {len(fold_cross)} folds' TRAIN/TEST composition is IDENTICAL between this Random Forest "
      f"run and the saved Logistic Regression run -- confirms a fair, apples-to-apples comparison "
      f"(only the estimator differs).")

import os
for f in ["labels_binary_preictal.parquet", "features_patient_scaled.parquet",
          "loso_fold_manifest.parquet", "features_chb24_forecasting.parquet",
          "features_chb12_forecasting.parquet"]:
    assert os.path.exists(output_dir / f)
for f in ["predictions.parquet", "fold_metrics.csv", "overall_horizon_summary.csv",
          "patient_horizon_summary.csv", "experiment_config.json"]:
    assert os.path.exists(lr_run_dir / f)
print("[OK] All previously existing outputs, including the LR baseline's outputs, remain untouched.")


[OK] predictions_df passes validate_predictions_df() (columns, value ranges).
[OK] All 57 folds match EEG_loso_evaluation_splitting.ipynb's saved manifest exactly (train/test counts, preictal/interictal counts) -- fold construction reproduced identically to the LR baseline.
[OK] All 57 folds' TRAIN/TEST composition is IDENTICAL between this Random Forest run and the saved Logistic Regression run -- confirms a fair, apples-to-apples comparison (only the estimator differs).
[OK] All previously existing outputs, including the LR baseline's outputs, remain untouched.


## 8. Per-fold metrics

`fold_seizure_event_metrics` (primary) and `fold_epoch_metrics` (secondary),
computed for every fold x model -- identical functions to the LR notebook.


In [8]:

fold_metrics_rows = []
for (pid, H, fold_id, sid, model_name), fold_df in predictions_df.groupby(
    ["patient_id", "horizon_minutes", "fold_id", "held_out_seizure_id", "model"]
):
    event_m = fold_seizure_event_metrics(fold_df)
    epoch_m = fold_epoch_metrics(fold_df)
    fold_metrics_rows.append({
        "patient_id": pid, "horizon_minutes": H, "fold_id": fold_id,
        "held_out_seizure_id": sid, "model": model_name,
        **event_m, **epoch_m,
    })
fold_metrics_df = pd.DataFrame(fold_metrics_rows)
fold_metrics_df = fold_metrics_df.merge(
    fold_summary_df[["patient_id", "horizon_minutes", "held_out_seizure_id",
                      "n_train", "train_preictal_count", "train_interictal_count",
                      "train_prevalence_preictal"]],
    on=["patient_id", "horizon_minutes", "held_out_seizure_id"], how="left",
)
print(f"fold_metrics_df: {fold_metrics_df.shape}")
print(fold_metrics_df[fold_metrics_df["model"] == "random_forest"][
    ["patient_id", "horizon_minutes", "held_out_seizure_id", "seizure_detected",
     "time_in_warning_sec", "false_alarms_per_hour", "auc_pr"]
].head(10).to_string(index=False))

rf_rows = fold_metrics_df[fold_metrics_df["model"] == "random_forest"]
n_zero_inter = int((rf_rows["n_test_interictal"] == 0).sum())
print(f"\n[REMINDER] Folds with 0 TEST interictal epochs (false_alarms_per_hour/auc_pr NaN by design): "
      f"{n_zero_inter} of {len(rf_rows)} random_forest fold-rows "
      f"(matches project_state.md Section 20.5's structural finding -- not a bug in this notebook, and "
      f"identical to the LR baseline's count since the folds are identical).")


fold_metrics_df: (114, 25)
patient_id  horizon_minutes held_out_seizure_id  seizure_detected  time_in_warning_sec  false_alarms_per_hour  auc_pr
     chb12                5     chb_s12_r06_sz0                 1                285.0                    NaN     NaN
     chb12                5     chb_s12_r08_sz0                 1                286.0                    NaN     NaN
     chb12                5     chb_s12_r23_sz0                 1                223.0                    NaN     NaN
     chb12                5     chb_s12_r38_sz0                 1                288.0                    NaN     NaN
     chb12                5     chb_s12_r42_sz0                 1                279.0                    NaN     NaN
     chb12               15     chb_s12_r06_sz0                 1                885.0                    NaN     NaN
     chb12               15     chb_s12_r08_sz0                 1                886.0                    NaN     NaN
     chb12               15  

## 9. Aggregate results: per patient + horizon, and overall by horizon

Pooled with the identical methodology used by the LR notebook (event counts
summed, epochs concatenated) -- never a naive mean of per-fold rates.


In [9]:

def summarize_group(event_df, epoch_pool_df, n_eligible_seizures):
    ev = pool_event_metrics(event_df)
    ep = pool_epoch_metrics(epoch_pool_df)
    return {"n_eligible_seizures": n_eligible_seizures, **ev, **ep}


patient_horizon_rows = []
for model_name in ["random_forest", "dummy_most_frequent"]:
    fm_model = fold_metrics_df[fold_metrics_df["model"] == model_name]
    pred_model = predictions_df[predictions_df["model"] == model_name]
    for pid in PATIENTS:
        for H in HORIZONS:
            ev_subset = fm_model[(fm_model["patient_id"] == pid) & (fm_model["horizon_minutes"] == H)]
            ep_subset = pred_model[(pred_model["patient_id"] == pid) & (pred_model["horizon_minutes"] == H)]
            if len(ev_subset) == 0:
                continue
            row = {"model": model_name, "patient_id": pid, "horizon_minutes": H}
            row.update(summarize_group(ev_subset, ep_subset, len(ev_subset)))
            patient_horizon_rows.append(row)
patient_horizon_summary_df = pd.DataFrame(patient_horizon_rows)

overall_horizon_rows = []
for model_name in ["random_forest", "dummy_most_frequent"]:
    fm_model = fold_metrics_df[fold_metrics_df["model"] == model_name]
    pred_model = predictions_df[predictions_df["model"] == model_name]
    for H in HORIZONS:
        ev_subset = fm_model[fm_model["horizon_minutes"] == H]
        ep_subset = pred_model[pred_model["horizon_minutes"] == H]
        row = {"model": model_name, "horizon_minutes": H}
        row.update(summarize_group(ev_subset, ep_subset, len(ev_subset)))
        overall_horizon_rows.append(row)
overall_horizon_summary_df = pd.DataFrame(overall_horizon_rows)

pd.set_option("display.width", 160)
display_cols = ["model", "patient_id", "horizon_minutes", "n_eligible_seizures", "seizure_sensitivity",
                 "false_alarms_per_hour", "time_in_warning_sec_mean", "auc_pr", "f1_preictal", "accuracy"]
print("=== PER PATIENT + HORIZON (primary + secondary metrics) ===")
print(patient_horizon_summary_df[display_cols].to_string(index=False))

print("\n=== OVERALL BY HORIZON (pooled across chb12 + chb24) ===")
overall_display_cols = ["model", "horizon_minutes", "n_eligible_seizures", "seizure_sensitivity",
                         "false_alarms_per_hour", "time_in_warning_sec_mean", "auc_pr", "f1_preictal", "accuracy"]
print(overall_horizon_summary_df[overall_display_cols].to_string(index=False))


=== PER PATIENT + HORIZON (primary + secondary metrics) ===
              model patient_id  horizon_minutes  n_eligible_seizures  seizure_sensitivity  false_alarms_per_hour  time_in_warning_sec_mean   auc_pr  f1_preictal  accuracy
      random_forest      chb12                5                    5             1.000000                    NaN                272.200000      NaN     1.000000  1.000000
      random_forest      chb12               15                    5             1.000000                    NaN                710.200000      NaN     0.991667  0.983471
      random_forest      chb12               30                    5             1.000000                    NaN               1082.200000      NaN     0.980609  0.961957
      random_forest      chb24                5                   14             0.785714              16.886228                200.272727 0.274211     0.302469  0.504386
      random_forest      chb24               15                   14             0.92

## 10. Sanity checks: class distribution, majority-class behaviour, confusion matrices

Same check the LR notebook runs against the earlier project's known failure
mode (project_state.md Section 18.1: 78.9% accuracy while detecting almost
nothing) -- if Random Forest predicts essentially only one class on TEST,
that is flagged explicitly here, not hidden behind a high accuracy number.


In [10]:

print("=== TEST-set predicted-positive rate (fraction of TEST epochs predicted PREICTAL) ===")
pos_rate_rows = []
for model_name in ["random_forest", "dummy_most_frequent"]:
    for H in HORIZONS:
        sub = predictions_df[(predictions_df["model"] == model_name) & (predictions_df["horizon_minutes"] == H)]
        pos_rate = sub["y_pred"].mean()
        true_pos_rate = sub["y_true"].mean()
        pos_rate_rows.append({"model": model_name, "horizon_minutes": H,
                               "predicted_positive_rate": pos_rate, "true_positive_rate": true_pos_rate})
pos_rate_df = pd.DataFrame(pos_rate_rows)
print(pos_rate_df.to_string(index=False))

flagged = pos_rate_df[
    (pos_rate_df["model"] == "random_forest")
    & ((pos_rate_df["predicted_positive_rate"] < 0.02) | (pos_rate_df["predicted_positive_rate"] > 0.98))
]
if len(flagged):
    print("\n[FLAG] Random Forest predicts essentially only one class at these horizons "
          "(likely repeating the old majority-class problem):")
    print(flagged.to_string(index=False))
else:
    print("\n[OK] Random Forest's predicted-positive rate is not degenerate (not <2% or >98%) "
          "at any horizon -- it is not trivially predicting only one class.")

print("\n=== Confusion matrices (pooled TEST epochs, per patient + horizon, Random Forest) ===")
confusion_rows = []
for pid in PATIENTS:
    for H in HORIZONS:
        sub = predictions_df[
            (predictions_df["model"] == "random_forest")
            & (predictions_df["patient_id"] == pid) & (predictions_df["horizon_minutes"] == H)
        ]
        cm = confusion_matrix(sub["y_true"], sub["y_pred"], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        confusion_rows.append({"patient_id": pid, "horizon_minutes": H,
                                "TN": tn, "FP": fp, "FN": fn, "TP": tp})
        print(f"{pid} H={H:2d}min:  TN={tn:4d}  FP={fp:4d}  FN={fn:4d}  TP={tp:4d}")
confusion_df = pd.DataFrame(confusion_rows)

print("\n=== PREICTAL probability distribution (Random Forest), by TRUE label ===")
for H in HORIZONS:
    sub = predictions_df[(predictions_df["model"] == "random_forest") & (predictions_df["horizon_minutes"] == H)]
    print(f"\nH={H} min:")
    print(sub.groupby("y_true")["y_prob"].describe()[["count", "mean", "std", "min", "50%", "max"]].to_string())

print("\n=== Out-of-bag-style overfitting signal: TRAIN accuracy vs TEST accuracy (Random Forest) ===")
overfit_rows = []
_seizure_inventory_check = seizure_inventory
for H in HORIZONS:
    fold_context_check = assign_fold_context(df, H)
    lab_col = f"binary_label_h{H}"
    for pid in PATIENTS:
        patient_df = df[df[PATIENT_COL] == pid].copy()
        eligible, _ = compute_seizure_eligibility(df, pid, H, seizure_inventory[pid])
        train_accs = []
        for fold_id, sid in enumerate(eligible, start=1):
            fold = build_loso_fold(patient_df, H, sid, fold_context_check, FEATURE_COLS)
            train_idx = fold["train_idx"]
            scaled_df, scaled_cols = fold["scaled_df"], fold["scaled_cols"]
            X_train = scaled_df.loc[train_idx, scaled_cols].to_numpy()
            y_train = (patient_df.loc[train_idx, lab_col] == "PREICTAL").astype(int).to_numpy()
            clf = RandomForestClassifier(**RF_PARAMS)
            clf.fit(X_train, y_train)
            train_accs.append(accuracy_score(y_train, clf.predict(X_train)))
        overfit_rows.append({
            "patient_id": pid, "horizon_minutes": H,
            "mean_train_accuracy": float(np.mean(train_accs)) if train_accs else np.nan,
        })
overfit_df = pd.DataFrame(overfit_rows)
overfit_merged = overfit_df.merge(
    patient_horizon_summary_df[patient_horizon_summary_df["model"] == "random_forest"][
        ["patient_id", "horizon_minutes", "accuracy"]
    ].rename(columns={"accuracy": "pooled_test_accuracy"}),
    on=["patient_id", "horizon_minutes"],
)
print(overfit_merged.to_string(index=False))
print("\n[NOTE] RandomForestClassifier grown to default max_depth=None fits TRAIN essentially perfectly "
      "(expected behaviour for unpruned trees); a large gap vs pooled TEST accuracy above is the "
      "textbook overfitting signature for tree ensembles and is discussed in the conclusion (Section 16).")


=== TEST-set predicted-positive rate (fraction of TEST epochs predicted PREICTAL) ===
              model  horizon_minutes  predicted_positive_rate  true_positive_rate
      random_forest                5                 0.496032            0.337302
      random_forest               15                 0.717587            0.571245
      random_forest               30                 0.790807            0.686679
dummy_most_frequent                5                 0.095238            0.337302
dummy_most_frequent               15                 0.667522            0.571245
dummy_most_frequent               30                 1.000000            0.686679

[OK] Random Forest's predicted-positive rate is not degenerate (not <2% or >98%) at any horizon -- it is not trivially predicting only one class.

=== Confusion matrices (pooled TEST epochs, per patient + horizon, Random Forest) ===
chb12 H= 5min:  TN=   0  FP=   0  FN=   0  TP=  48
chb12 H=15min:  TN=   0  FP=   0  FN=   2  TP= 119
chb1

patient_id  horizon_minutes  mean_train_accuracy  pooled_test_accuracy
     chb12                5             1.000000              1.000000
     chb24                5             0.995997              0.504386
     chb12               15             0.997185              0.983471
     chb24               15             1.000000              0.465046
     chb12               30             0.996902              0.961957
     chb24               30             0.999651              0.564626

[NOTE] RandomForestClassifier grown to default max_depth=None fits TRAIN essentially perfectly (expected behaviour for unpruned trees); a large gap vs pooled TEST accuracy above is the textbook overfitting signature for tree ensembles and is discussed in the conclusion (Section 16).


## 11. Naive-baseline comparison: Random Forest vs. always-majority-class

`DummyClassifier(strategy="most_frequent")`, fit on the exact same per-fold
TRAIN labels as the LR notebook's naive baseline. By construction it can
only ever get `seizure_sensitivity` > 0 in folds where PREICTAL happened to
be the TRAIN-majority class; comparing the two side by side is the direct
test for whether Random Forest is doing better than exploiting class
imbalance.


In [11]:

compare_cols = ["horizon_minutes", "seizure_sensitivity", "false_alarms_per_hour",
                "time_in_warning_sec_mean", "auc_pr", "accuracy", "f1_preictal"]
rf_overall = overall_horizon_summary_df[overall_horizon_summary_df["model"] == "random_forest"][compare_cols].set_index("horizon_minutes")
dummy_overall = overall_horizon_summary_df[overall_horizon_summary_df["model"] == "dummy_most_frequent"][compare_cols].set_index("horizon_minutes")
comparison_df = rf_overall.join(dummy_overall, lsuffix="_rf", rsuffix="_dummy")
print("=== Random Forest vs. naive majority-class baseline (pooled overall, by horizon) ===")
print(comparison_df.to_string())


=== Random Forest vs. naive majority-class baseline (pooled overall, by horizon) ===
                 seizure_sensitivity_rf  false_alarms_per_hour_rf  time_in_warning_sec_mean_rf  auc_pr_rf  accuracy_rf  f1_preictal_rf  seizure_sensitivity_dummy  false_alarms_per_hour_dummy  time_in_warning_sec_mean_dummy  auc_pr_dummy  accuracy_dummy  f1_preictal_dummy
horizon_minutes                                                                                                                                                                                                                                                                
5                              0.842105                 16.886228                   222.750000   0.528692     0.551587        0.461905                   0.263158                     0.000000                      272.200000      0.524416        0.757937           0.440367
15                             0.947368                 18.323353                   716.833333   0.

## 12. Feature importance interpretation

Random Forest's `feature_importances_` (mean decrease in Gini impurity) is
the tree-ensemble analogue of the LR notebook's standardized coefficients --
both are computed from a per-fold, per-patient scaled feature matrix, so
values are comparable *within* a fold's model. Summarized here as
mean/std/rank-stability across the 57 fold-level models, not as one single
"the" importance.

**These are NOT causal biomarkers**, exactly as stated in the LR notebook --
descriptive of this baseline on this specific (small, 2-patient) sample
only. Unlike LR coefficients, `feature_importances_` are always >= 0 and sum
to 1 per fold, so there is no "sign" to report -- instead, rank stability
(how often a feature lands in the per-fold top 5) is reported as the
importance analogue of LR's sign-consistency check.


In [12]:

importance_summary_overall = (
    importances_df.groupby("feature")["importance"]
    .agg(mean_importance="mean", std_importance="std")
    .sort_values("mean_importance", ascending=False)
    .reset_index()
)
print("=== Feature importance summary, pooled across ALL 57 folds (both patients, all horizons) ===")
print(importance_summary_overall.to_string(index=False))

importance_summary_by_horizon = (
    importances_df.groupby(["horizon_minutes", "feature"])["importance"]
    .agg(mean_importance="mean", std_importance="std")
    .reset_index()
)
importance_summary_by_patient_horizon = (
    importances_df.groupby(["patient_id", "horizon_minutes", "feature"])["importance"]
    .agg(mean_importance="mean", std_importance="std")
    .reset_index()
)

top5_per_fold = (
    importances_df.sort_values(["patient_id", "horizon_minutes", "fold_id", "importance"], ascending=[True, True, True, False])
    .groupby(["patient_id", "horizon_minutes", "fold_id"])
    .head(5)
)
top5_rate = (
    top5_per_fold.groupby("feature").size() / importances_df["fold_id"].nunique()
    if False else top5_per_fold.groupby("feature").size() / 57
).rename("pct_folds_in_top5").reset_index()
importance_summary_overall = importance_summary_overall.merge(top5_rate, on="feature", how="left")
importance_summary_overall["pct_folds_in_top5"] = importance_summary_overall["pct_folds_in_top5"].fillna(0.0)

print("\nTop 5 features by mean importance:")
print(importance_summary_overall.head(5)[["feature", "mean_importance", "std_importance", "pct_folds_in_top5"]].to_string(index=False))

print(f"\n[NOTE] Rank-stability check (analogue of LR's sign-consistency check): fraction of the 57 "
      f"fold-level models in which each feature ranks in that fold's own top-5 by importance.")
print(importance_summary_overall[["feature", "pct_folds_in_top5"]].sort_values("pct_folds_in_top5", ascending=False).to_string(index=False))


=== Feature importance summary, pooled across ALL 57 folds (both patients, all horizons) ===
          feature  mean_importance  std_importance
          entropy         0.114357        0.057710
         rel_beta         0.113134        0.047775
             beta         0.108034        0.049437
 delta_beta_ratio         0.097291        0.023304
theta_alpha_ratio         0.078026        0.037739
        rel_alpha         0.069557        0.029925
        rel_delta         0.067091        0.028134
            alpha         0.063422        0.013733
        rel_theta         0.052987        0.013308
            delta         0.051574        0.030602
            theta         0.049359        0.022950
              var         0.045842        0.023975
              std         0.045544        0.020951
              rms         0.043783        0.022214

Top 5 features by mean importance:
          feature  mean_importance  std_importance  pct_folds_in_top5
          entropy         0.114357  

## 13. Diagnostic plots

In [13]:

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- PR curves, pooled overall per horizon ---
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 4), sharey=True)
for ax, H in zip(axes, HORIZONS):
    for model_name, style in [("random_forest", "-"), ("dummy_most_frequent", "--")]:
        sub = predictions_df[(predictions_df["model"] == model_name) & (predictions_df["horizon_minutes"] == H)]
        if sub["y_true"].nunique() < 2:
            continue
        precision, recall, _ = precision_recall_curve(sub["y_true"], sub["y_prob"])
        ap = average_precision_score(sub["y_true"], sub["y_prob"])
        ax.plot(recall, precision, style, label=f"{model_name} (AP={ap:.3f})")
    ax.set_title(f"H = {H} min (pooled, both patients)")
    ax.set_xlabel("Recall")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Precision")
fig.suptitle("Precision-Recall curves -- pooled TEST epochs (overall, per horizon), Random Forest")
fig.tight_layout()
fig.savefig(plot_dir / "pr_curves_overall_by_horizon.png", dpi=150)
plt.close(fig)

# --- Confusion matrix heatmaps, per patient + horizon (Random Forest) ---
fig, axes = plt.subplots(len(PATIENTS), len(HORIZONS), figsize=(4 * len(HORIZONS), 4 * len(PATIENTS)))
for i, pid in enumerate(PATIENTS):
    for j, H in enumerate(HORIZONS):
        ax = axes[i, j]
        sub = predictions_df[
            (predictions_df["model"] == "random_forest")
            & (predictions_df["patient_id"] == pid) & (predictions_df["horizon_minutes"] == H)
        ]
        cm = confusion_matrix(sub["y_true"], sub["y_pred"], labels=[0, 1])
        im = ax.imshow(cm, cmap="Greens")
        for r in range(2):
            for c in range(2):
                ax.text(c, r, str(cm[r, c]), ha="center", va="center")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["INTERICTAL", "PREICTAL"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["INTERICTAL", "PREICTAL"])
        ax.set_title(f"{pid}, H={H}min")
        if j == 0:
            ax.set_ylabel("True")
        if i == len(PATIENTS) - 1:
            ax.set_xlabel("Predicted")
fig.suptitle("Confusion matrices (pooled TEST epochs, Random Forest)")
fig.tight_layout()
fig.savefig(plot_dir / "confusion_matrices_rf.png", dpi=150)
plt.close(fig)

# --- PREICTAL probability distributions, per horizon, split by true label ---
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 4), sharey=True)
for ax, H in zip(axes, HORIZONS):
    sub = predictions_df[(predictions_df["model"] == "random_forest") & (predictions_df["horizon_minutes"] == H)]
    ax.hist(sub.loc[sub["y_true"] == 0, "y_prob"], bins=20, alpha=0.6, label="true INTERICTAL", density=True)
    ax.hist(sub.loc[sub["y_true"] == 1, "y_prob"], bins=20, alpha=0.6, label="true PREICTAL", density=True)
    ax.axvline(THRESHOLD, color="black", linestyle=":", label=f"threshold={THRESHOLD}")
    ax.set_title(f"H = {H} min")
    ax.set_xlabel("Predicted P(PREICTAL)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Density")
fig.suptitle("Predicted PREICTAL probability by true label (Random Forest, pooled)")
fig.tight_layout()
fig.savefig(plot_dir / "probability_distributions.png", dpi=150)
plt.close(fig)

# --- Primary-metric comparison across horizons, per patient (RF vs dummy) ---
metric_names = ["seizure_sensitivity", "false_alarms_per_hour", "auc_pr"]
fig, axes = plt.subplots(len(PATIENTS), len(metric_names), figsize=(4.5 * len(metric_names), 4 * len(PATIENTS)))
for i, pid in enumerate(PATIENTS):
    for j, metric in enumerate(metric_names):
        ax = axes[i, j]
        for model_name, marker in [("random_forest", "o-"), ("dummy_most_frequent", "s--")]:
            sub = patient_horizon_summary_df[
                (patient_horizon_summary_df["model"] == model_name) & (patient_horizon_summary_df["patient_id"] == pid)
            ].sort_values("horizon_minutes")
            ax.plot(sub["horizon_minutes"], sub[metric], marker, label=model_name)
        ax.set_title(f"{pid}: {metric}")
        ax.set_xlabel("Horizon (min)")
        ax.set_xticks(HORIZONS)
        if j == 0:
            ax.legend(fontsize=7)
fig.suptitle("Primary metrics across horizons, per patient (Random Forest)")
fig.tight_layout()
fig.savefig(plot_dir / "primary_metrics_by_horizon.png", dpi=150)
plt.close(fig)

# --- Feature importance bar chart, pooled overall ---
fig, ax = plt.subplots(figsize=(8, 6))
plot_imp = importance_summary_overall.sort_values("mean_importance", ascending=True)
ax.barh(plot_imp["feature"], plot_imp["mean_importance"], xerr=plot_imp["std_importance"])
ax.set_xlabel("Mean feature importance (Gini, pooled across 57 folds)")
ax.set_title("Random Forest feature importance -- pooled across all folds")
fig.tight_layout()
fig.savefig(plot_dir / "feature_importance_overall.png", dpi=150)
plt.close(fig)

print("[OK] Saved 5 diagnostic plot files to", plot_dir)
for p in sorted(plot_dir.glob("*.png")):
    print(" -", p.name)


[OK] Saved 5 diagnostic plot files to C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\modeling\random_forest_loso_binary\plots
 - confusion_matrices_rf.png
 - feature_importance_overall.png
 - pr_curves_overall_by_horizon.png
 - primary_metrics_by_horizon.png
 - probability_distributions.png


## 14. Save outputs

New, clearly-named directory: `outputs/modeling/random_forest_loso_binary/`.
Nothing existing, including `outputs/modeling/logistic_loso_binary/`, is
overwritten.


In [14]:

predictions_df.to_parquet(run_dir / "predictions.parquet", index=False)
predictions_df.to_csv(run_dir / "predictions.csv", index=False)

fold_metrics_df.to_csv(run_dir / "fold_metrics.csv", index=False)
patient_horizon_summary_df.to_csv(run_dir / "patient_horizon_summary.csv", index=False)
overall_horizon_summary_df.to_csv(run_dir / "overall_horizon_summary.csv", index=False)

importances_df.to_csv(run_dir / "feature_importances_long.csv", index=False)
importance_summary_overall.to_csv(run_dir / "feature_importances_summary_overall.csv", index=False)
importance_summary_by_horizon.to_csv(run_dir / "feature_importances_summary_by_horizon.csv", index=False)
importance_summary_by_patient_horizon.to_csv(run_dir / "feature_importances_summary_by_patient_horizon.csv", index=False)

import json as _json
experiment_config = {
    "task": "Binary PREICTAL vs INTERICTAL detection, patient-specific LOSO, Random Forest baseline",
    "model": "RandomForestClassifier",
    "model_params": RF_PARAMS,
    "naive_baseline": "DummyClassifier(strategy='most_frequent')",
    "random_state": RANDOM_STATE,
    "decision_threshold": THRESHOLD,
    "threshold_tuned_on_test": False,
    "hyperparameter_tuning_performed": False,
    "horizons_minutes": HORIZONS,
    "patients": PATIENTS,
    "patients_pooled_across": False,
    "feature_cols": FEATURE_COLS,
    "epoch_sec": EPOCH_SEC,
    "n_folds_total": int(n_folds_built),
    "source_labels_table": str(in_path),
    "source_loso_manifest_crosscheck": str(manifest_path),
    "reference_lr_run": str(lr_run_dir),
    "normalization": "per-fold, per-patient StandardScaler fit on TRAIN rows only (reused verbatim from EEG_patient_normalization.ipynb / EEG_loso_evaluation_splitting.ipynb / EEG_logistic_loso_binary.ipynb)",
    "metrics_framework_source": "EEG_seizure_prediction_metrics.ipynb / EEG_logistic_loso_binary.ipynb",
    "class_imbalance_handling": "class_weight='balanced' -- RF analogue of the LR baseline's class_weight='balanced', both computed from the fold's own TRAIN y at fit() time",
}
with open(run_dir / "experiment_config.json", "w") as f:
    _json.dump(experiment_config, f, indent=2, default=str)

print("Saved to", run_dir, ":")
for p in sorted(run_dir.glob("*")):
    if p.is_file():
        print(" -", p.name)

import os as _os
for f in ["labels_binary_preictal.parquet", "features_patient_scaled.parquet",
          "loso_fold_manifest.parquet", "features_chb24_forecasting.parquet",
          "features_chb12_forecasting.parquet"]:
    assert _os.path.exists(output_dir / f)
for f in ["predictions.parquet", "fold_metrics.csv", "overall_horizon_summary.csv",
          "patient_horizon_summary.csv", "experiment_config.json"]:
    assert _os.path.exists(lr_run_dir / f)
print("\n[OK] All previously existing outputs, including the LR baseline's outputs, remain untouched.")


Saved to C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\modeling\random_forest_loso_binary :
 - experiment_config.json
 - feature_importances_long.csv
 - feature_importances_summary_by_horizon.csv
 - feature_importances_summary_by_patient_horizon.csv
 - feature_importances_summary_overall.csv
 - fold_metrics.csv
 - overall_horizon_summary.csv
 - patient_horizon_summary.csv
 - predictions.csv
 - predictions.parquet

[OK] All previously existing outputs, including the LR baseline's outputs, remain untouched.


## 15. Logistic Regression vs. Random Forest comparison

Reads the Logistic Regression numbers directly from
`outputs/modeling/logistic_loso_binary/overall_horizon_summary.csv` and
`patient_horizon_summary.csv` (the actual saved outputs from that already-
validated run) -- **not** hardcoded from any prior report. The RF numbers
are from this run's own `overall_horizon_summary_df` / `patient_horizon_summary_df`
in memory (already verified in Section 7 to be evaluated on byte-for-byte
identical folds to the LR run).


In [15]:

lr_overall_saved = pd.read_csv(lr_run_dir / "overall_horizon_summary.csv")
lr_overall_saved = lr_overall_saved[lr_overall_saved["model"] == "logistic_regression"].copy()
lr_overall_saved["model"] = "logistic_regression"

rf_overall_saved = overall_horizon_summary_df[overall_horizon_summary_df["model"] == "random_forest"].copy()

combined_overall = pd.concat([lr_overall_saved, rf_overall_saved], ignore_index=True, sort=False)
combined_overall["detected_over_eligible"] = (
    combined_overall["n_seizures_detected"].astype(int).astype(str) + "/" + combined_overall["n_folds"].astype(int).astype(str)
)

print("=" * 100)
print("LOGISTIC REGRESSION vs RANDOM FOREST -- POOLED OVERALL, BY HORIZON")
print("=" * 100)
comp_cols = ["horizon_minutes", "model", "seizure_sensitivity", "detected_over_eligible",
             "false_alarms_per_hour", "auc_pr"]
print(combined_overall.sort_values(["horizon_minutes", "model"])[comp_cols].to_string(index=False))

lr_patient_saved = pd.read_csv(lr_run_dir / "patient_horizon_summary.csv")
lr_patient_saved = lr_patient_saved[lr_patient_saved["model"] == "logistic_regression"].copy()
rf_patient_saved = patient_horizon_summary_df[patient_horizon_summary_df["model"] == "random_forest"].copy()
combined_patient = pd.concat([lr_patient_saved, rf_patient_saved], ignore_index=True, sort=False)
combined_patient["detected_over_eligible"] = (
    combined_patient["n_seizures_detected"].astype(int).astype(str) + "/" + combined_patient["n_folds"].astype(int).astype(str)
)

print("\n" + "=" * 100)
print("LOGISTIC REGRESSION vs RANDOM FOREST -- PER PATIENT + HORIZON")
print("=" * 100)
comp_patient_cols = ["patient_id", "horizon_minutes", "model", "seizure_sensitivity",
                      "detected_over_eligible", "false_alarms_per_hour", "auc_pr"]
print(combined_patient.sort_values(["patient_id", "horizon_minutes", "model"])[comp_patient_cols].to_string(index=False))

comparison_table_path = run_dir / "lr_vs_rf_comparison_overall.csv"
combined_overall.sort_values(["horizon_minutes", "model"])[comp_cols].to_csv(comparison_table_path, index=False)
comparison_patient_path = run_dir / "lr_vs_rf_comparison_by_patient.csv"
combined_patient.sort_values(["patient_id", "horizon_minutes", "model"])[comp_patient_cols].to_csv(comparison_patient_path, index=False)
print(f"\n[OK] Saved comparison tables to {comparison_table_path.name} and {comparison_patient_path.name}")

print("\n[LIMITATION, still present] CHB12 folds have 0 TEST interictal epochs at every horizon "
      "(project_state.md Section 20.5) -- false_alarms_per_hour and auc_pr are undefined (NaN, not 0 or a "
      "fabricated value) for CHB12 for BOTH models, at both the per-patient and (since CHB12 contributes 0 "
      "interictal hours/epochs to the pooled sums) implicitly within the 'overall' pooled rows too. The "
      "overall FA/hour and AUC-PR numbers above are therefore driven entirely by CHB24.")


LOGISTIC REGRESSION vs RANDOM FOREST -- POOLED OVERALL, BY HORIZON
 horizon_minutes               model  seizure_sensitivity detected_over_eligible  false_alarms_per_hour   auc_pr
               5 logistic_regression             0.947368                  18/19              13.652695 0.535829
               5       random_forest             0.842105                  16/19              16.886228 0.528692
              15 logistic_regression             0.947368                  18/19              13.293413 0.650486
              15       random_forest             0.947368                  18/19              18.323353 0.731450
              30 logistic_regression             0.947368                  18/19              12.574850 0.736408
              30       random_forest             0.947368                  18/19              15.808383 0.767989

LOGISTIC REGRESSION vs RANDOM FOREST -- PER PATIENT + HORIZON
patient_id  horizon_minutes               model  seizure_sensitivity detected_o

## 16. Scientific interpretation

In [16]:

def _get(df, H, model):
    row = df[(df["horizon_minutes"] == H) & (df["model"] == model)]
    return row.iloc[0] if len(row) else None

lr5, rf5 = _get(combined_overall, 5, "logistic_regression"), _get(combined_overall, 5, "random_forest")
lr15, rf15 = _get(combined_overall, 15, "logistic_regression"), _get(combined_overall, 15, "random_forest")
lr30, rf30 = _get(combined_overall, 30, "logistic_regression"), _get(combined_overall, 30, "random_forest")

def _fmt(x, nd=3):
    return "NaN" if pd.isna(x) else f"{x:.{nd}f}"

auc_deltas = {H: (_get(combined_overall, H, "random_forest")["auc_pr"] - _get(combined_overall, H, "logistic_regression")["auc_pr"]) for H in HORIZONS}
fa_deltas = {H: (_get(combined_overall, H, "random_forest")["false_alarms_per_hour"] - _get(combined_overall, H, "logistic_regression")["false_alarms_per_hour"]) for H in HORIZONS}
sens_deltas = {H: (_get(combined_overall, H, "random_forest")["seizure_sensitivity"] - _get(combined_overall, H, "logistic_regression")["seizure_sensitivity"]) for H in HORIZONS}

chb24_rf = combined_patient[(combined_patient["patient_id"] == "chb24") & (combined_patient["model"] == "random_forest")]
chb12_rf = combined_patient[(combined_patient["patient_id"] == "chb12") & (combined_patient["model"] == "random_forest")]

overfit_gap = overfit_merged.assign(gap=lambda d: d["mean_train_accuracy"] - d["pooled_test_accuracy"])

h30_train_prev = fold_summary_df[fold_summary_df["horizon_minutes"] == 30]["train_prevalence_preictal"]
h30_majority_leans_positive = int((h30_train_prev > 0.5).sum())

summary_lines = []
summary_lines.append("### 1. Does RF improve AUC-PR over LR at any horizon?")
for H in HORIZONS:
    summary_lines.append(f"  H={H:2d}min: LR AUC-PR={_fmt(_get(combined_overall,H,'logistic_regression')['auc_pr'])}, "
                          f"RF AUC-PR={_fmt(_get(combined_overall,H,'random_forest')['auc_pr'])}, delta={_fmt(auc_deltas[H])}")

summary_lines.append("\n### 2. Does RF reduce false alarms/hour where FA/hour is measurable?")
for H in HORIZONS:
    summary_lines.append(f"  H={H:2d}min: LR FA/hr={_fmt(_get(combined_overall,H,'logistic_regression')['false_alarms_per_hour'])}, "
                          f"RF FA/hr={_fmt(_get(combined_overall,H,'random_forest')['false_alarms_per_hour'])}, delta={_fmt(fa_deltas[H])}")

summary_lines.append("\n### 3. Does RF maintain seizure sensitivity?")
for H in HORIZONS:
    summary_lines.append(f"  H={H:2d}min: LR sens={_fmt(_get(combined_overall,H,'logistic_regression')['seizure_sensitivity'])}, "
                          f"RF sens={_fmt(_get(combined_overall,H,'random_forest')['seizure_sensitivity'])}, delta={_fmt(sens_deltas[H])}")

summary_lines.append("\n### 4. Is any apparent improvement driven mainly by CHB24?")
summary_lines.append(f"  CHB12 contributes 0 TEST interictal epochs at every horizon (Section 15 limitation) -- "
                      f"CHB12's false_alarms_per_hour/auc_pr are NaN for both models. Any pooled FA/hour or "
                      f"AUC-PR difference between RF and LR is therefore mechanically driven entirely by CHB24 "
                      f"({len(chb24_rf)} horizon-rows), not a genuine 2-patient effect.")

summary_lines.append("\n### 5. Does RF meaningfully beat the simple LR baseline, or is the representation the bottleneck?")
max_abs_auc_delta = max(abs(v) for v in auc_deltas.values())
summary_lines.append(f"  Largest |AUC-PR delta| across horizons: {max_abs_auc_delta:.3f}. Both models use the "
                      f"identical 14 qEEG features and identical folds -- if this delta is small, it suggests "
                      f"the qEEG feature representation itself (not the linear-vs-tree-ensemble choice) is the "
                      f"binding constraint on further gains, consistent with project_state.md's roadmap toward "
                      f"raw-EEG/CNN representations (Section 9) as the next lever, rather than further classical-"
                      f"ML tuning.")

summary_lines.append("\n### 6. Are there signs of overfitting?")
summary_lines.append(f"  Mean TRAIN accuracy vs pooled TEST accuracy (Random Forest, default max_depth=None):")
for _, r in overfit_gap.iterrows():
    summary_lines.append(f"    {r['patient_id']} H={int(r['horizon_minutes']):2d}min: TRAIN acc={r['mean_train_accuracy']:.3f}, "
                          f"TEST acc={r['pooled_test_accuracy']:.3f}, gap={r['gap']:.3f}")
summary_lines.append(f"  A consistently large gap (TRAIN >> TEST) is the expected signature of unpruned "
                      f"(max_depth=None) trees memorizing TRAIN -- present here per the table above, and an "
                      f"argument for depth/leaf-size tuning as a *future* (not this-task) step.")

summary_lines.append("\n### 7. Does the H=30 class-imbalance / majority-baseline caveat remain important?")
summary_lines.append(f"  Yes. {h30_majority_leans_positive}/{len(h30_train_prev)} folds at H=30min still have TRAIN "
                      f"PREICTAL prevalence > 0.5 (same structural property noted for LR in project_state.md "
                      f"Section 22.2). The dummy_most_frequent baseline at H=30 in Section 11 above shows the same "
                      f"pattern already documented for LR -- high sensitivity and low FA/hour from a constant-"
                      f"positive-leaning predictor is not automatically evidence of a time-varying detector, "
                      f"for RF any more than it was for LR.")

print("\n".join(summary_lines))

print("\n\n[CAUTION] All numbers above come from 2 patients (chb12, chb24) and 19 total eligible seizures "
      "across all three horizons combined -- per-fold and even per-patient metrics are individually noisy. "
      "None of this should be overclaimed as a generalizable RF-vs-LR verdict beyond this specific 2-patient "
      "sample and this specific qEEG feature set.")


### 1. Does RF improve AUC-PR over LR at any horizon?
  H= 5min: LR AUC-PR=0.536, RF AUC-PR=0.529, delta=-0.007
  H=15min: LR AUC-PR=0.650, RF AUC-PR=0.731, delta=0.081
  H=30min: LR AUC-PR=0.736, RF AUC-PR=0.768, delta=0.032

### 2. Does RF reduce false alarms/hour where FA/hour is measurable?
  H= 5min: LR FA/hr=13.653, RF FA/hr=16.886, delta=3.234
  H=15min: LR FA/hr=13.293, RF FA/hr=18.323, delta=5.030
  H=30min: LR FA/hr=12.575, RF FA/hr=15.808, delta=3.234

### 3. Does RF maintain seizure sensitivity?
  H= 5min: LR sens=0.947, RF sens=0.842, delta=-0.105
  H=15min: LR sens=0.947, RF sens=0.947, delta=-0.000
  H=30min: LR sens=0.947, RF sens=0.947, delta=-0.000

### 4. Is any apparent improvement driven mainly by CHB24?
  CHB12 contributes 0 TEST interictal epochs at every horizon (Section 15 limitation) -- CHB12's false_alarms_per_hour/auc_pr are NaN for both models. Any pooled FA/hour or AUC-PR difference between RF and LR is therefore mechanically driven entirely by CHB24 (3 ho